## Notebook to read in a visualize DRAGONS (and other GMIMS) data
### A. Ordog, Nov 3, 2026

In [ ]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord, Galactic
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


### Read in the FITS file, extract data cube and header
Note: in this example, looking at the CLEAN Faraday depth cube, but there are many other data products in the DRAGONS data directory to explore (see README for details)

In [ ]:
hdu = fits.open('../../dragons_download/data/faraday_synthesis/dragons_FDF_clean_tot_Kgal.car.fits')

fdf = hdu[0].data
print(fdf.shape)
print('')

hdr = hdu[0].header
print(repr(hdr))

### Make an array of the Faraday depth channels using WCS:

In [ ]:
wcs = WCS(hdr)
print(wcs)
print('')

FDvals = wcs.all_pix2world(0,0,range(fdf.shape[0]),0)[2]
print(FDvals)


### Make a map of a single Faraday depth slice:

In [ ]:
#######################
FD_pick = 5 # rad/m^2
#######################

FDidx = abs(FD_pick - FDvals).argmin()
print(f'Using {FDvals[FDidx]} rad/m^2')

fig = plt.figure(figsize=(12,6))

gs = gridspec.GridSpec(ncols=1, nrows=1, figure=fig)
ax = fig.add_subplot(gs[0], projection=wcs.celestial)

ax.imshow(fdf[FDidx], origin='lower', vmin=0, vmax=1)

### This time zoom in on a specified region

In [ ]:
#######################
FD_pick = -7 # rad/m^2
lpick = 137 # deg
bpick = 7 # deg
width = 20 # deg
#######################

FDidx = abs(FD_pick - FDvals).argmin()
print(f'Using {FDvals[FDidx]} rad/m^2')

crange1 = SkyCoord([lpick+width/2, lpick-width/2], [bpick,bpick], frame=Galactic, unit=(u.deg, u.deg))
crange2 = SkyCoord([lpick,lpick], [bpick-width/2, bpick+width/2], frame=Galactic, unit=(u.deg, u.deg))

fig = plt.figure(figsize=(6,6))

gs = gridspec.GridSpec(ncols=1, nrows=1, figure=fig)
ax = fig.add_subplot(gs[0], projection=wcs.celestial)

ax.imshow(fdf[FDidx], origin='lower', vmin=0, vmax=1)

ax.set_ylim(wcs.dropaxis(dropax=2).world_to_pixel(crange2)[1])
ax.set_xlim(wcs.dropaxis(dropax=2).world_to_pixel(crange1)[0])

### Now plot the Faraday depth spectrum at a chosen position

In [ ]:
#######################
FD_pick = -7 # rad/m^2
lpick = 137 # deg
bpick = 7 # deg
width = 20 # deg
#######################

l_idx, b_idx = wcs.dropaxis(dropax=2).all_world2pix(lpick, bpick, 0)
l_idx = int(np.floor(l_idx))
b_idx = int(np.floor(b_idx))
print(l_idx,b_idx)

crange1 = SkyCoord([lpick+width/2, lpick-width/2], [bpick,bpick], frame=Galactic, unit=(u.deg, u.deg))
crange2 = SkyCoord([lpick,lpick], [bpick-width/2, bpick+width/2], frame=Galactic, unit=(u.deg, u.deg))

fig = plt.figure(figsize=(12,4))

gs = gridspec.GridSpec(ncols=2, nrows=1, figure=fig)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], projection=wcs.celestial)

ax1.plot(FDvals, fdf[:,b_idx,l_idx])
ax1.set_xlim(-50,50)
ax1.set_ylim(0,)

ax2.imshow(fdf[FDidx], origin='lower', vmin=0, vmax=1)
ax2.scatter(l_idx,b_idx)

ax2.set_ylim(wcs.dropaxis(dropax=2).world_to_pixel(crange2)[1])
ax2.set_xlim(wcs.dropaxis(dropax=2).world_to_pixel(crange1)[0])